## LLM Implementation in LangGraph -- Basic

1. Call LangGraph, LangChain
2. OpenAI module from LangChain
3. Build a State for messages
4. Build a Chat Loop
5. Graph for the simple logic in LangGraph

In [33]:
from typing import TypedDict, List
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv

In [34]:
load_dotenv()

True

In [35]:
CHAT_MODEL = "gpt-5-mini"

In [60]:
class AgentState(TypedDict):
    system_message: SystemMessage
    human_messages: List[HumanMessage]
    llm: ChatOpenAI
    ai_response: ChatOpenAI

In [71]:
def initiate_llm(state: AgentState) -> AgentState:
    """
    Call a LLM client to initiate via LangChain Openai
    """
    
    state["llm"] = ChatOpenAI(model=CHAT_MODEL)

    return state

In [72]:
def invoke_llm(state: AgentState) -> AgentState:
    """
    Process system and human messages and invoke LLM response
    """
    messages = [state["system_message"], state["human_messages"]]
    response = state["llm"].invoke(messages)

    state["ai_response"] = response

    return state

In [73]:
graph = StateGraph(AgentState)

graph.add_node("llm_initiator", initiate_llm)
graph.add_node("invoke_llm", invoke_llm) 

graph.add_edge(START, "llm_initiator")
graph.add_edge("llm_initiator", "invoke_llm")
graph.add_edge("invoke_llm", END)

app = graph.compile()

In [74]:
result = app.invoke(
    {
        "system_message": SystemMessage("You are a helpful assistant."),
        "human_messages": HumanMessage(input())
    }
)

result

 hey


{'system_message': SystemMessage(content='You are a helpful assistant.', additional_kwargs={}, response_metadata={}),
 'human_messages': HumanMessage(content='hey', additional_kwargs={}, response_metadata={}),
 'llm': ChatOpenAI(profile={'max_input_tokens': 272000, 'max_output_tokens': 128000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x110e30500>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x110e307d0>, root_client=<openai.OpenAI object at 0x110e30320>, root_async_client=<openai.AsyncOpenAI object at 0x110e305f0>, model_name='gpt-5-min

In [75]:
result["ai_response"].content

'Hey — how can I help you today? Need info, help writing something, coding, or just chatting?'